# PoC 2b: Scaled RAG + Surprise Scoring

Scaled-up version: Azure `text-embedding-3-small`, ~200-300 papers (full PDFs + abstracts), arxiv + Semantic Scholar, proper chunking.

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import AzureOpenAI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
EMBED_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")

r = client.embeddings.create(input=["test"], model=EMBED_MODEL)
print(f"LLM: {MODEL}, Embeddings: {EMBED_MODEL} (dim={len(r.data[0].embedding)})")

## 1. Fetch Papers — arxiv + Semantic Scholar

In [ ]:
import arxiv

ARXIV_QUERIES = [
    "spectrum monitoring machine learning",
    "spectrum monitoring deep learning",
    "6G wireless communications",
    "6G network architecture future",
    "test measurement automation RF",
    "signal processing cognitive radio",
    "dynamic spectrum access",
    "electronic warfare spectrum sensing",
    "LEO satellite communications RF",
    "Open RAN disaggregated network",
    "Open RAN security 5G",
    "AI native wireless network",
    "RF fingerprinting identification",
    "spectrum sharing coexistence",
    "reconfigurable intelligent surface RIS",
    "joint sensing communication",
    "digital twin wireless network",
    "quantum communications RF",
    "terahertz communications",
    "non-terrestrial network NTN satellite",
]
MAX_PER_QUERY = 15

papers = []
seen_ids = set()

for i, query in enumerate(ARXIV_QUERIES):
    search = arxiv.Search(query=query, max_results=MAX_PER_QUERY, sort_by=arxiv.SortCriterion.Relevance)
    count = 0
    for result in search.results():
        if result.entry_id not in seen_ids:
            seen_ids.add(result.entry_id)
            papers.append({
                "id": result.entry_id,
                "title": result.title,
                "abstract": result.summary,
                "published": result.published.strftime("%Y-%m-%d"),
                "pdf_url": result.pdf_url,
                "source": "arxiv",
                "query": query,
            })
            count += 1
    print(f"  [{i+1}/{len(ARXIV_QUERIES)}] '{query}' → {count} new papers")

print(f"\narxiv total: {len(papers)} unique papers")

In [ ]:
S2_QUERIES = [
    "spectrum monitoring autonomous",
    "6G technology roadmap",
    "test measurement instrument automation",
    "electronic warfare signal intelligence",
    "satellite spectrum sensing",
    "Open RAN deployment challenges",
    "AI wireless resource management",
    "RF machine learning classification",
    "future wireless communications 2030",
    "spectrum policy regulation sharing",
]
S2_URL = "https://api.semanticscholar.org/graph/v1/paper/search"
S2_FIELDS = "title,abstract,year,externalIds,citationCount"

for i, query in enumerate(S2_QUERIES):
    try:
        resp = requests.get(S2_URL, params={"query": query, "limit": 20, "fields": S2_FIELDS}, timeout=10)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        count = 0
        for p in data:
            if not p.get("abstract"):
                continue
            s2_id = f"s2:{p.get('paperId', '')}"
            if s2_id in seen_ids:
                continue
            seen_ids.add(s2_id)
            arxiv_id = (p.get("externalIds") or {}).get("ArXiv")
            papers.append({
                "id": s2_id,
                "title": p["title"],
                "abstract": p["abstract"],
                "published": str(p.get("year", "unknown")),
                "pdf_url": f"https://arxiv.org/pdf/{arxiv_id}" if arxiv_id else None,
                "source": "semantic_scholar",
                "query": query,
            })
            count += 1
        print(f"  [{i+1}/{len(S2_QUERIES)}] '{query}' → {count} new papers")
        time.sleep(1)  # rate limit: 100 req / 5 min
    except Exception as e:
        print(f"  [{i+1}/{len(S2_QUERIES)}] '{query}' → FAILED: {e}")

print(f"\nTotal papers: {len(papers)} (arxiv + Semantic Scholar)")
pd.DataFrame(papers)["source"].value_counts()

## 2. Download PDFs + Extract Full Text

Download available PDFs and extract text with PyMuPDF. Falls back to abstract if PDF fails.

In [ ]:
import pymupdf
from pathlib import Path

PDF_DIR = Path("../data/pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

def download_pdf(url: str, path: Path, timeout: int = 30) -> bool:
    try:
        resp = requests.get(url, timeout=timeout, stream=True)
        resp.raise_for_status()
        with open(path, "wb") as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
        return True
    except Exception:
        return False

def extract_text_from_pdf(path: Path, max_pages: int = 15) -> str | None:
    try:
        doc = pymupdf.open(str(path))
        text = ""
        for page in doc[:max_pages]:
            text += page.get_text()
        doc.close()
        return text.strip() if len(text.strip()) > 200 else None
    except Exception:
        return None

pdf_success = 0
pdf_fail = 0
abstract_only = 0

for i, paper in enumerate(papers):
    if i % 50 == 0:
        print(f"Processing {i}/{len(papers)}...")

    if paper.get("pdf_url"):
        safe_name = paper["id"].replace("/", "_").replace(":", "_")[-40:] + ".pdf"
        pdf_path = PDF_DIR / safe_name

        if pdf_path.exists():
            text = extract_text_from_pdf(pdf_path)
        else:
            if download_pdf(paper["pdf_url"], pdf_path):
                text = extract_text_from_pdf(pdf_path)
            else:
                text = None

        if text:
            paper["full_text"] = text
            pdf_success += 1
            continue
        else:
            pdf_fail += 1

    paper["full_text"] = f"Title: {paper['title']}\n\nAbstract: {paper['abstract']}"
    abstract_only += 1

print(f"\nPDF extracted: {pdf_success}, PDF failed (abstract fallback): {pdf_fail}, Abstract only: {abstract_only}")

## 3. Chunk + Build RAG Index (Azure Embeddings + ChromaDB)

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

Settings.embed_model = AzureOpenAIEmbedding(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_deployment=EMBED_MODEL,
)
Settings.llm = None

documents = [
    Document(
        text=p["full_text"],
        metadata={"title": p["title"], "published": p["published"], "source": p["source"], "id": p["id"]},
    )
    for p in papers
]

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection("papers_scaled")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_documents(
    documents,
    vector_store=vector_store,
    transformations=[splitter],
    show_progress=True,
)

print(f"\nIndexed {len(documents)} documents → {chroma_collection.count()} chunks")

## 4. Extract Technology Drivers (RAG-grounded)

In [ ]:
from src.models import DriverExtractionResult

FORESIGHT_QUERIES = [
    "What emerging technologies will transform spectrum monitoring by 2035?",
    "What are the biggest disruptions coming to wireless test and measurement?",
    "How will AI change RF signal processing and cognitive radio?",
    "What new sensing modalities could replace traditional spectrum analyzers?",
    "What geopolitical or regulatory shifts could reshape the wireless industry?",
    "How will satellite constellations change spectrum management?",
    "What role will digital twins play in wireless network planning?",
    "What quantum technologies could impact RF communications?",
    "How will Open RAN change the competitive landscape for T&M vendors?",
    "What are the most surprising or non-obvious technology trends in wireless?",
]

all_context_chunks = []
retriever = index.as_retriever(similarity_top_k=6)

for q in FORESIGHT_QUERIES:
    nodes = retriever.retrieve(q)
    for node in nodes:
        all_context_chunks.append(node.get_text())

unique_chunks = list(set(all_context_chunks))
print(f"Retrieved {len(unique_chunks)} unique context chunks from {len(FORESIGHT_QUERIES)} queries")

# Split into batches to avoid token limits
BATCH_SIZE = 15
all_drivers = []

for batch_start in range(0, len(unique_chunks), BATCH_SIZE):
    batch = unique_chunks[batch_start:batch_start + BATCH_SIZE]
    context_block = "\n\n---\n\n".join(batch)

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a technology foresight analyst specializing in wireless communications, "
                "spectrum management, and test & measurement (T&M) for a company like Rohde & Schwarz. "
                "Extract technology drivers from the provided research context. "
                "A driver is a technology trend, capability, or external force that could significantly "
                "shape the industry in the next 5-10 years. "
                "Include BOTH obvious/mainstream drivers AND surprising/non-obvious ones. "
                "Aim for 10-15 drivers per batch. Be specific — 'AI-based RF fingerprinting' not just 'AI'."
            )},
            {"role": "user", "content": f"Extract technology drivers from this research context:\n\n{context_block}"},
        ],
        response_format=DriverExtractionResult,
        temperature=0.4,
    )
    batch_drivers = response.choices[0].message.parsed.drivers
    all_drivers.extend(batch_drivers)
    print(f"  Batch {batch_start//BATCH_SIZE + 1}: {len(batch_drivers)} drivers")

print(f"\nTotal raw drivers: {len(all_drivers)}")

## 5. Deduplicate Drivers (Semantic Similarity)

Multiple batches produce overlapping drivers. Cluster by embedding similarity and merge.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

def azure_embed(texts: list[str]) -> np.ndarray:
    """Embed texts in batches via Azure OpenAI."""
    all_embeds = []
    for i in range(0, len(texts), 16):
        batch = texts[i:i+16]
        resp = client.embeddings.create(input=batch, model=EMBED_MODEL)
        all_embeds.extend([d.embedding for d in resp.data])
    return np.array(all_embeds)

driver_texts = [f"{d.name}: {d.description}" for d in all_drivers]
driver_embeddings = azure_embed(driver_texts)

sim_matrix = cosine_similarity(driver_embeddings)
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.15,  # merge if cosine distance < 0.15
    metric="cosine",
    linkage="average",
)
labels = clustering.fit_predict(driver_embeddings)

# Pick the driver with the longest description per cluster as representative
deduped_drivers = []
deduped_embeddings = []
for cluster_id in range(labels.max() + 1):
    members = [i for i, l in enumerate(labels) if l == cluster_id]
    best = max(members, key=lambda i: len(all_drivers[i].description))
    deduped_drivers.append(all_drivers[best])
    deduped_embeddings.append(driver_embeddings[best])

deduped_embeddings = np.array(deduped_embeddings)
print(f"Deduplicated: {len(all_drivers)} raw → {len(deduped_drivers)} unique drivers")

## 6. Surprise Scoring (Azure Embeddings)

In [ ]:
centroid = deduped_embeddings.mean(axis=0, keepdims=True)
similarities = cosine_similarity(deduped_embeddings, centroid).flatten()
surprise_scores = 1 - similarities

driver_df = pd.DataFrame([d.model_dump() for d in deduped_drivers])
driver_df["surprise_score"] = surprise_scores
driver_df = driver_df.sort_values("surprise_score", ascending=False)

print(f"Score range: {surprise_scores.min():.4f} — {surprise_scores.max():.4f}")
print(f"Median: {np.median(surprise_scores):.4f}\n")
print("Top 10 potential blind spots:")
driver_df[["name", "steep_category", "surprise_score"]].head(10)

## 7. Visualization

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(deduped_drivers) * 0.35)))

# Bar chart
plot_df = driver_df.sort_values("surprise_score", ascending=True)
colors = plt.cm.RdYlGn_r(plot_df["surprise_score"] / plot_df["surprise_score"].max())
axes[0].barh(range(len(plot_df)), plot_df["surprise_score"], color=colors)
axes[0].set_yticks(range(len(plot_df)))
axes[0].set_yticklabels(plot_df["name"], fontsize=8)
axes[0].set_xlabel("Surprise Score")
axes[0].set_title("Driver Surprise Scores\n(high = potential blind spot)")
axes[0].axvline(x=plot_df["surprise_score"].median(), color="gray", linestyle="--", alpha=0.5, label="median")
axes[0].legend()

# PCA scatter with STEEP coloring
pca = PCA(n_components=2)
coords = pca.fit_transform(deduped_embeddings)
centroid_2d = pca.transform(centroid)

steep_cats = driver_df["steep_category"].unique()
steep_colors = {cat: plt.cm.tab10(i) for i, cat in enumerate(steep_cats)}

for cat in steep_cats:
    mask = driver_df["steep_category"] == cat
    idx = driver_df.index[mask]
    pos = [driver_df.index.get_loc(i) for i in idx]
    axes[1].scatter(coords[pos, 0], coords[pos, 1], c=[steep_colors[cat]], label=cat, s=80, edgecolors="black", linewidths=0.5)

axes[1].scatter(centroid_2d[0, 0], centroid_2d[0, 1], c="blue", marker="X", s=200, label="centroid", zorder=5)

for i, row in driver_df.iterrows():
    pos = driver_df.index.get_loc(i)
    axes[1].annotate(row["name"], (coords[pos, 0], coords[pos, 1]), fontsize=6, alpha=0.8)

axes[1].set_title("Driver Embedding Space (PCA)\nColored by STEEP category")
axes[1].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

## Full Driver Table

In [ ]:
driver_df[["name", "description", "steep_category", "trl_low", "trl_high", "surprise_score"]].round(4)